In [172]:
user_query = input("Enter your query:").lower()
print("You entered:", user_query)


You entered: hi


In [173]:
import re

def determine_self_contained(query: str) -> str:
    # Exact contextual replies
    FOLLOW_UP_EXACT = {
    "yes","no","nah","yup","yeah","ok","okay","sure",
    "continue",
    "go ahead",
    "do it",
    "that",
    "this",
    "it",
    "same",
    "exactly",
    "correct",
    "right",
    "indeed",
    "absolutely",
    "definitely",
    "perhaps",
    "maybe",
    "possibly"
}

    FOLLOW_UP_STARTERS = {
    "can you clarify",
    "what do you mean",
    "could you explain",
    "can you explain",
    "please elaborate",
    "elaborate",
    "tell me more",
    "explain more",
    "go deeper",
    "expand on that",
    "can you expand",
    "clarify",
    "continue",
    "continue please",
    "what about that",
    "what about this",
    "why is that",
    "how so",
}

    QUESTION_WORDS = {
    "what",
    "what's",
    "how",
    "why",
    "when",
    "where",
    "which",
    "who",
    "whose",
    "whom",
}
    GREETING_WORDS = {"hi", "hello", "hey", "greetings","yoo","yo","sup","wassup","howdy","hola","bonjour","ciao","salut"}
    PRONOUNS = {"it", "that", "this", "them", "those", "these"}
    DOMAIN_WORDS = {"jwt", "docker", "lambda", "aws", "python", "course"}   #depends on each topic :)

    q = query.lower().strip()
    q = re.sub(r"[^\w\s]", "", q)
    words = q.split()

    if not words:
        return "follow-up"
    
    if any(w in GREETING_WORDS for w in words):
        return "self-contained"

    if q in FOLLOW_UP_EXACT or any(q.startswith(p) for p in FOLLOW_UP_STARTERS):
        return "follow-up"

    score = 0.0

    has_pronoun = any(w in PRONOUNS for w in words)
    has_domain = any(w in DOMAIN_WORDS for w in words)

    if has_pronoun and not has_domain:
        score -= 3
    elif has_pronoun and has_domain:
        score -= 1
    if words[0] in QUESTION_WORDS:
        score += 2
    elif any(w in QUESTION_WORDS for w in words):
        score += 0.5

    if has_domain:
        score += 2

    length_bonus = min(len(words) / 5, 2.0) 
    score += length_bonus

    if len(words) <= 2 and not has_domain:
        score -= 2

    return "self-contained" if score >= 1 else "follow-up"

In [174]:
determine_self_contained(user_query)

'self-contained'

In [175]:
from dataclasses import dataclass


@dataclass
class IntentResult:
    intent: str
    confidence: float
    evidence: list[str]

In [176]:
INTENTS = [
    "clarify_needed",
    "simple_fact_question",
    "domain_question",
    "troubleshoot_issue",
    "follow_up_question",
    "reflection_request",
    "redo_assessment",
    "goal_setting",
    "out_of_scope",
    "chit_chat"
]

In [177]:
scores = {
    intent: 0
    for intent in INTENTS
}

In [178]:
CHIT_CHAT = {

    # Greetings
    "hi",
    "hii",
    "hiii",
    "hello",
    "helo",
    "hey",
    "heyy",
    "heyyy",
    "yo",
    "sup",
    "wassup",
    "what's up",
    "whats up",
    "good morning",
    "good afternoon",
    "good evening",
    "good night",
    "morning",
    "afternoon",
    "evening",

    # Introductions
    "my name is",
    "i am",
    "i'm",
    "this is",

    # Small talk
    "how are you",
    "how are you doing",
    "how's it going",
    "how have you been",
    "how do you do",
    "what's going on",
    "what are you doing",
    "how is your day",
    "how was your day",
    "hope you're doing well",
    "hope you are doing well",

    # Acknowledgements
    "okay",
    "ok",
    "kk",
    "k",
    "alright",
    "all right",
    "got it",
    "gotcha",
    "understood",
    "makes sense",
    "fair enough",
    "cool",
    "nice",
    "great",
    "awesome",
    "perfect",
    "fine",
    "sure",

    # Gratitude
    "thanks",
    "thank you",
    "thankyou",
    "thanks a lot",
    "thank you so much",
    "many thanks",
    "much appreciated",
    "appreciate it",
    "really appreciate it",
    "thanks buddy",
    "thanks bro",

    # Positive feedback
    "good job",
    "well done",
    "great job",
    "excellent",
    "amazing",
    "awesome work",
    "brilliant",
    "fantastic",
    "that helped",
    "that was helpful",

    # Farewells
    "bye",
    "bye bye",
    "goodbye",
    "see you",
    "see ya",
    "see you later",
    "talk later",
    "catch you later",
    "take care",
    "have a nice day",
    "have a good day",
    "good night",
    "see you tomorrow",

    # Casual reactions
    "lol",
    "lmao",
    "haha",
    "hahaha",
    "hehe",
    "wow",
    "oh",
    "oh wow",
    "nice one",
    "interesting",

    # Politeness
    "please",
    "excuse me",
    "sorry",
    "my apologies",
    "pardon",

    # Conversational fillers
    "hmm",
    "hmmm",
    "uh",
    "uhh",
    "umm",
    "ummm",
    "okay then",
    "right",
    "yep",
    "yeah",
    "yup",
    "nah",
    "nope",

    # Meta conversation
    "who are you",
    "what are you",
    "what can you do",
    "tell me about yourself",
    "introduce yourself",
}

if user_query in CHIT_CHAT and determine_self_contained(user_query) == "self-contained":
    scores["chit_chat"] += 20
else:
    scores["follow_up_question"] += 5

In [179]:
print(scores["chit_chat"])

20


In [180]:
REDO = [
 
    # Explicit redo
    "redo assessment",
    "redo the assessment",
    "redo my assessment",
    "redo test",
    "redo quiz",
    "redo exam",
    "redo evaluation",

    # Restart
    "restart assessment",
    "restart the assessment",
    "restart my assessment",
    "restart test",
    "restart quiz",
    "restart exam",
    "restart evaluation",

    # Retake
    "retake assessment",
    "retake the assessment",
    "retake test",
    "retake quiz",
    "retake exam",
    "retake evaluation",
    "take again",
    "take it again",
    "attempt again",
    "try again",

    # Start over
    "start over",
    "start again",
    "begin again",
    "restart from beginning",
    "restart from scratch",
    "start from scratch",
    "begin from scratch",

    # Reset
    "reset assessment",
    "reset the assessment",
    "reset my assessment",
    "reset test",
    "reset quiz",
    "reset exam",
    "clear assessment",
    "wipe assessment",

    # Reattempt
    "reattempt assessment",
    "reattempt test",
    "reattempt quiz",
    "give another attempt",
    "take another attempt",

    # Informal
    "can i redo",
    "can i retake",
    "can i restart",
    "can i try again",
    "can i do it again",
    "i want to redo",
    "i want to retake",
    "i want to restart",
    "i want to try again",
    "let me try again",
    "let me retake",
    "let me restart",

    # Planning phrasing
    "i would like to retake",
    "i would like to restart",
    "i need to retake",
    "i need to restart",
    "help me restart",
    "help me retake",

    # Failure-driven
    "i failed can i retake",
    "i failed can i try again",
    "can i attempt once more",
    "can i give the assessment again",
    "can i take the assessment again",

    # One-word cases
    "redo",
    "retake",
    "restart",
    "reset"
]

if any(x in user_query for x in REDO) and determine_self_contained(user_query) == "self-contained":
    scores["redo_assessment"] += 15

if any(x in user_query for x in REDO) and determine_self_contained(user_query) != "self-contained":
    scores["clarify_needed"]+= 5


In [181]:
GOAL_PATTERNS = [

    # Planning
    "help me plan",
    "can you plan",
    "create a plan",
    "make a plan",
    "plan for me",
    "help me create a plan",
    "help me make a plan",

    # Roadmaps
    "create a roadmap",
    "make a roadmap",
    "roadmap",
    "learning roadmap",
    "career roadmap",
    "study roadmap",

    # Learning path
    "learning path",
    "learning journey",
    "study path",
    "career path",
    "path to learn",

    # Preparation
    "help me prepare",
    "prepare me",
    "how should i prepare",
    "how can i prepare",
    "where should i start",
    "how should i start",
    "how do i start",
    "what should i learn first",

    # Learning goals
    "i want to learn",
    "i want to study",
    "i want to master",
    "i want to improve",
    "i want to understand",
    "i need to learn",
    "i need to improve",
    "i want to practice",

    # Career goals
    "i want to become",
    "i want to switch to",
    "i want to transition to",
    "career goal",
    "career transition",
    "career change",

    # Study plans
    "study plan",
    "study schedule",
    "learning schedule",
    "preparation strategy",
    "learning strategy",

    # Guidance
    "guide me",
    "mentor me",
    "help me get started",
    "help me learn",
    "show me the path",
    "show me the roadmap",

    # Skill development
    "what skills should i learn",
    "what should i focus on",
    "what should i study",
    "what should i practice",
    "what do i need to learn",
    "what skills do i need",

    # Certification preparation
    "help me crack",
    "help me clear",
    "prepare for interview",
    "prepare for certification",
    "prepare for exam",

    # Goal statements
    "my goal",
    "my objective",
    "my target",
    "my aim",
    "my plan"
]

if any(x in user_query for x in GOAL_PATTERNS):
    scores["goal_setting"] += 15
# else:
#     scores["reflection_request"] += 5

In [182]:
TROUBLE = [

    # Generic problem words
    "error",
    "errors",
    "issue",
    "issues",
    "problem",
    "problems",
    "bug",
    "bugs",
    "fault",
    "failure",

    # Failure states
    "fail",
    "fails",
    "failed",
    "failing",
    "failure",
    "crash",
    "crashes",
    "crashed",
    "crashing",
    "broken",
    "stuck",

    # Unable cases
    "unable",
    "cannot",
    "can't",
    "could not",
    "couldn't",
    "won't",
    "doesn't",
    "does not",

    # Working state
    "not working",
    "isn't working",
    "stopped working",
    "doesn't work",
    "does not work",
    "not functioning",
    "not responding",

    # Exceptions
    "exception",
    "stacktrace",
    "traceback",
    "runtime error",
    "compile error",
    "syntax error",

    # Performance
    "timeout",
    "timed out",
    "slow",
    "latency",
    "hanging",
    "freeze",
    "freezing",

    # Connectivity
    "connection refused",
    "connection error",
    "network error",
    "network issue",
    "disconnected",

    # Authentication
    "unauthorized",
    "forbidden",
    "permission denied",
    "access denied",
    "authentication failed",

    # Unexpected behavior
    "unexpected",
    "incorrect",
    "wrong output",
    "wrong result",
    "unexpected behavior",

    # Help seeking
    "help",
    "fix",
    "resolve",
    "troubleshoot",
    "debug",

    # Emotional indicators
    "confused",
    "stuck on",
    "don't understand",
    "doesn't make sense",

    # Status
    "failed to",
    "unable to",
    "cannot",
    "can't get",
    "can't access",
    "can't connect"
]

for word in TROUBLE:
    if word in user_query:
        scores["troubleshoot_issue"] += 20

In [183]:
REFLECTION = [

    # Explicit reflection
    "reflect",
    "reflection",
    "reflect on",
    "self reflection",

    # Learning extraction
    "what did i learn",
    "what have i learned",
    "what did i understand",
    "what did i miss",
    "what should i take away",
    "what is the takeaway",
    "key takeaway",
    "lesson learned",
    "lessons learned",

    # Performance review
    "how did i do",
    "how am i doing",
    "how well did i do",
    "evaluate myself",
    "evaluate my performance",
    "assess my performance",
    "review my performance",
    "rate my performance",

    # Improvement
    "what can i improve",
    "how can i improve",
    "where can i improve",
    "what should i improve",
    "what are my weaknesses",
    "what are my strengths",
    "what skills should i improve",

    # Retrospective
    "what went wrong",
    "what went well",
    "what could have been better",
    "what should i have done",
    "what mistakes did i make",
    "where did i go wrong",

    # Understanding
    "why did i fail",
    "why did i struggle",
    "why was this difficult",
    "why couldn't i solve this",

    # Self evaluation
    "how did i perform",
    "how was my performance",
    "am i improving",
    "have i improved",
    "how much have i learned",

    # Coaching style
    "give me feedback",
    "provide feedback",
    "analyze my performance",
    "review my answers",
    "analyze my approach",
    "evaluate my approach",
    "assess my understanding",

    # Future learning
    "what should i do differently",
    "what should i focus on next",
    "what should i learn next",
    "what are my next steps"
]

if any(x in user_query for x in REFLECTION):
    scores["reflection_request"] += 15

In [184]:
FOLLOWUPS = [
    "why",
    "how",
    "what about",
    "also",
    "and",
    "then",
    "okay but",
    "tell me more",
    "explain more"
]

if any(user_query.startswith(x) for x in FOLLOWUPS):
    scores["follow_up_question"] += 10

In [185]:
print("Scores:", scores)

Scores: {'clarify_needed': 0, 'simple_fact_question': 0, 'domain_question': 0, 'troubleshoot_issue': 0, 'follow_up_question': 0, 'reflection_request': 0, 'redo_assessment': 0, 'goal_setting': 0, 'out_of_scope': 0, 'chit_chat': 20}
